# Grupo N.º 2  
## Trabajo Grupal 3  
 ### Integrantes:
 - Héctor Ramos Vera 
 - David Sebastian Leon Guaman
 - Polk Brando Vernaza Quiñonez
 


### Análisis de Datos: Monitoreo de Pacientes en UCI y Predicción de Mortalidad (15 000 registros)
- El propósito de este conjunto de datos, centrado en el Análisis de Datos: Monitoreo de Pacientes en UCI y Predicción de Mortalidad, es proporcionar una base sólida para mejorar la toma de decisiones críticas en entornos hospitalarios mediante el uso de inteligencia de datos

### Instalación y configuración de Kaggle para la descarga del dataset

In [4]:
!pip install kaggle


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


- Descagar el dataset de de forma remota

In [5]:
!kaggle datasets download jayjoshi37/icu-patient-monitoring-and-mortality-prediction

Dataset URL: https://www.kaggle.com/datasets/jayjoshi37/icu-patient-monitoring-and-mortality-prediction
License(s): CC0-1.0




  0%|          | 0.00/707k [00:00<?, ?B/s]
100%|██████████| 707k/707k [00:00<00:00, 2.04MB/s]
100%|██████████| 707k/707k [00:00<00:00, 2.03MB/s]


## 1 Selección del dataset -
- Descromprimir el archivo zip

In [6]:
import zipfile
with zipfile.ZipFile("icu-patient-monitoring-and-mortality-prediction.zip", 'r') as zip_ref:
    zip_ref.extractall("icu_dataset")

- Importar las librerias 

In [7]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report
from sklearn.impute import SimpleImputer
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline

- Instalación de la librería KaggleHub

In [8]:
!pip install kagglehub


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Importación de la librería KaggleHub

In [9]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

c:\Users\POLK\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


- Carga del dataset en un DataFrame

In [10]:
df=pd.read_csv('icu_dataset/ICU_Patient_Monitoring_Mortality_Prediction_15000.csv')

In [11]:
df

,patient_id,age,gender,admission_type,comorbidity_score,heart_rate_mean,heart_rate_std,heart_rate_max,heart_rate_min,systolic_bp_mean,...,glucose_mean,lactate_mean,urine_output_total,ventilation_required,vasopressor_used,length_of_stay_days,apache_score,sofa_score,sepsis_flag,mortality_label
0,PAT000001,32,Male,Urgent,3.75,136.06,19.64,162.01,127.16,104.04,...,73.71,7.77,4245.99,0,0,6.32,15.65,10.50,0,0
1,PAT000002,46,Male,Urgent,2.91,108.95,7.79,124.18,94.79,131.05,...,78.36,5.06,1267.36,0,1,29.00,33.29,6.09,0,1
2,PAT000003,87,Male,Urgent,6.84,95.21,7.44,117.54,89.35,171.84,...,168.41,1.89,4863.13,1,1,26.95,25.93,18.44,0,1
3,PAT000004,21,Male,Emergency,1.96,63.62,11.51,82.22,51.84,164.59,...,214.40,1.06,4940.99,1,0,1.16,33.54,14.14,0,1
4,PAT000005,21,Male,Urgent,7.71,65.92,12.17,74.98,39.34,146.10,...,201.33,5.28,4492.46,0,0,21.68,31.63,11.23,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14995,PAT014996,27,Female,Elective,0.02,100.21,15.97,105.34,79.25,123.65,...,217.13,3.22,1034.24,1,0,13.41,13.89,8.30,0,0
14996,PAT014997,68,Male,Emergency,8.39,88.72,13.34,100.69,77.45,148.53,...,171.81,5.17,989.13,0,1,14.36,38.71,1.03,0,0
14997,PAT014998,24,Female,Elective,9.06,83.32,23.45,109.28,56.80,133.29,...,201.38,2.23,3007.17,0,1,18.94,19.35,12.90,1,0
14998,PAT014999,82,Female,Emergency,3.60,86.64,23.59,95.87,67.42,137.37,...,203.62,2.99,1402.62,0,1,17.93,38.89,12.27,0,0


- Identificación de valores únicos en la variable de mortalidad

In [12]:
df['mortality_label'].unique()

array([0, 1])

- Distribución de la variable de mortalidad

In [13]:
df.groupby('mortality_label').size()

mortality_label
0    11582
1     3418
dtype: int64

In [14]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   patient_id             15000 non-null  str    
 1   age                    15000 non-null  int64  
 2   gender                 15000 non-null  str    
 3   admission_type         15000 non-null  str    
 4   comorbidity_score      15000 non-null  float64
 5   heart_rate_mean        15000 non-null  float64
 6   heart_rate_std         15000 non-null  float64
 7   heart_rate_max         15000 non-null  float64
 8   heart_rate_min         15000 non-null  float64
 9   systolic_bp_mean       15000 non-null  float64
 10  systolic_bp_std        15000 non-null  float64
 11  respiratory_rate_mean  15000 non-null  float64
 12  spo2_mean              15000 non-null  float64
 13  temperature_mean       15000 non-null  float64
 14  glucose_mean           15000 non-null  float64
 15  lactate_mean 

### 2  Procesamiento de datos 
- Limpieza y Transformación de Datos


In [15]:
if df ['mortality_label'].dtype == 'object':
  le = LabelEncoder()
  df['mortality_label'] = le.fit_transform(df['mortality_label'])

  df = pd.get_dummies(df)

In [16]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   patient_id             15000 non-null  str    
 1   age                    15000 non-null  int64  
 2   gender                 15000 non-null  str    
 3   admission_type         15000 non-null  str    
 4   comorbidity_score      15000 non-null  float64
 5   heart_rate_mean        15000 non-null  float64
 6   heart_rate_std         15000 non-null  float64
 7   heart_rate_max         15000 non-null  float64
 8   heart_rate_min         15000 non-null  float64
 9   systolic_bp_mean       15000 non-null  float64
 10  systolic_bp_std        15000 non-null  float64
 11  respiratory_rate_mean  15000 non-null  float64
 12  spo2_mean              15000 non-null  float64
 13  temperature_mean       15000 non-null  float64
 14  glucose_mean           15000 non-null  float64
 15  lactate_mean 

- Detección y reporte de valores nulos

In [17]:
nulls   = df.isnull().sum()
null_pct = (nulls / len(df) * 100).round(2)
null_report = pd.DataFrame({'Nulos': nulls, 'Porcentaje (%)': null_pct})
null_report = null_report[null_report['Nulos'] > 0].sort_values('Porcentaje (%)', ascending=False)

print(f'Columnas con valores nulos: {len(null_report)} de {df.shape[1]}')
display(null_report if not null_report.empty else 'Sin valores nulos')

Columnas con valores nulos: 0 de 24


'Sin valores nulos'

- Separar columnas numéricas y categóricas 

In [18]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
print(f'\nColumnas numéricas  : {len(num_cols)}')
print(f'Columnas categóricas: {len(cat_cols)}')

if num_cols:
    imp_num = SimpleImputer(strategy='median')
    df[num_cols] = imp_num.fit_transform(df[num_cols])

if cat_cols:
    imp_cat = SimpleImputer(strategy='most_frequent')
    df[cat_cols] = imp_cat.fit_transform(df[cat_cols])

print('\n Imputación completada (mediana para numéricas, moda para categóricas).')

C:\Users\POLK\AppData\Local\Temp\ipykernel_22996\4246366318.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()



Columnas numéricas  : 21
Columnas categóricas: 3

 Imputación completada (mediana para numéricas, moda para categóricas).


- Eliminar duplicados

In [19]:
n_before = len(df)
df.drop_duplicates(inplace=True)
print(f' Duplicados eliminados: {n_before - len(df)}')

 Duplicados eliminados: 0


- Identificación de valores atípicos usando el rango intercuartílico (IQR)

In [20]:
outlier_report = {}
for col in num_cols:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    n_out = ((df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)).sum()
    if n_out > 0:
        outlier_report[col] = n_out

outlier_df = pd.DataFrame.from_dict(outlier_report, orient='index', columns=['Outliers (IQR)'])
print(f'\nColumnas con outliers detectados: {len(outlier_df)}')
display(outlier_df.sort_values('Outliers (IQR)', ascending=False).head(10))


Columnas con outliers detectados: 3


,Outliers (IQR)
vasopressor_used,3677
mortality_label,3418
sepsis_flag,3029


- Detección automática de la columna objetivo (mortalidad)

In [21]:
TARGET_CANDIDATES = [
    'mortality', 'death', 'survived', 'outcome', 'hospital_death',
    'in_hospital_death', 'mortality_flag', 'deceased'
]
target_col = None
for c in df.columns:
    if c.lower().replace(' ', '_') in TARGET_CANDIDATES:
        target_col = c
        break

- primera columna binaria encontrada al final del df

In [22]:
if target_col is None:
    for c in reversed(df.columns):
        if df[c].nunique() == 2:
            target_col = c
            break

if target_col:
    print(f'\n Columna objetivo detectada: "{target_col}"')
    print(df[target_col].value_counts().rename_axis('Clase').reset_index(name='Conteo').to_string(index=False))
else:
    print('\n No se detectó columna objetivo binaria. Ajusta target_col manualmente.')


 Columna objetivo detectada: "mortality_label"
 Clase  Conteo
   0.0   11582
   1.0    3418


- Codificación de categóricas para ML

In [23]:
df_ml = df.copy()
le    = LabelEncoder()
for col in cat_cols:
    if col != target_col:
        df_ml[col] = le.fit_transform(df_ml[col].astype(str))

print(f'\n Limpieza y transformación completadas. Shape final: {df.shape}')


 Limpieza y transformación completadas. Shape final: (15000, 24)
